# 7-4절 연습 문제 풀이

이 노트북은 7-4절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch07/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 7-3/7-4절 공통 - MNIST 오토인코더
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
DATA_ROOT = '../../downloads'

def mnist(train=True):
    return datasets.MNIST(root=DATA_ROOT, train=train, download=True,
                          transform=transforms.ToTensor())

class MNISTAutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                                     nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                     nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z).view(-1, 1, 28, 28), z

def train_ae(model, epochs=5, lr=1e-3, noise=0.0):
    loader = DataLoader(mnist(), batch_size=128, shuffle=True)
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train(); tot = n = 0
        for x, _ in loader:
            x = x.to(device)
            inp = (x + torch.randn_like(x) * noise).clamp(0, 1) if noise else x
            out, _ = model(inp)
            loss = criterion(out, x)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * len(x); n += len(x)
        print(f'  {e}/{epochs} 복원 손실 {tot / n:.5f}')
    return model

## 연습 7-13

잠재 벡터의 크기를 10으로 늘려 오토인코더와 두 전이 학습 모델을 학습한 후 결과가 어떻게 바뀌는지 확인해 보자.

In [ ]:
def transfer_experiment(latent_dim, n_train=1000, n_valid=1000, n_test=8000):
    torch.manual_seed(SEED)
    ae = train_ae(MNISTAutoEncoder(latent_dim), epochs=5)
    train_set, test_set = mnist(), mnist(False)
    g = torch.Generator().manual_seed(SEED)
    perm = torch.randperm(len(train_set), generator=g)
    tr = Subset(train_set, perm[:n_train].tolist())
    va = Subset(train_set, perm[n_train:n_train + n_valid].tolist())
    te = Subset(test_set, list(range(min(n_test, len(test_set)))))
    results = {}
    for mode in ('특징 추출', '미세 조정'):
        torch.manual_seed(SEED)
        import copy
        enc = copy.deepcopy(ae.encoder)
        if mode == '특징 추출':
            for p_ in enc.parameters(): p_.requires_grad = False
        clf = nn.Sequential(enc, nn.Linear(latent_dim, 10)).to(device)
        opt = torch.optim.Adam([p_ for p_ in clf.parameters() if p_.requires_grad], lr=1e-3)
        crit = nn.CrossEntropyLoss()
        for _ in range(10):
            clf.train()
            for x, y in DataLoader(tr, batch_size=64, shuffle=True):
                loss = crit(clf(x.to(device)), y.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        clf.eval(); correct = n = 0
        with torch.no_grad():
            for x, y in DataLoader(te, batch_size=256):
                correct += (clf(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
        results[mode] = correct / n * 100
    return results

print('잠재 벡터 크기 10')
for mode, acc in transfer_experiment(10).items():
    print(f'  {mode}: {acc:.2f}%')

잠재 벡터를 2에서 10으로 늘리면 인코더가 담는 정보가 크게 늘어 두 방식 모두 정확도가 오른다. 특히 **특징 추출 방식의 향상 폭이 크다**. 고정된 인코더의 표현력이 곧 성능 상한이기 때문이다.

## 연습 7-14

잠재 벡터의 크기가 2인 원래의 전이 학습 예제에서 훈련, 검증, 평가 데이터셋의 샘플 수를 각각 1,000개, 1,000개, 8,000개로 조정하고 모델을 학습한 후 결과가 어떻게 바뀌는지를 확인해 보자. 만약 특징 추출 방식과 미세 조정 방식 모델의 정확도 차이가 벌어진다면 그 이유가 무엇인지 설명해 보자.

In [ ]:
print('잠재 벡터 크기 2 / 훈련 1000, 검증 1000, 평가 8000')
for mode, acc in transfer_experiment(2).items():
    print(f'  {mode}: {acc:.2f}%')

**정확도 차이가 벌어지는 이유**

잠재 벡터가 2차원이면 인코더가 784차원 이미지를 단 두 숫자로 압축한다. 이 표현은 복원에는 쓸 만해도 **10개 클래스를 구분하기에는 정보가 턱없이 부족**하다.

- **특징 추출**: 인코더가 고정되어 이 부족한 2차원 표현을 그대로 써야 하므로 성능이 낮다.
- **미세 조정**: 인코더까지 분류 목적에 맞게 다시 학습되어 같은 2차원이라도 '분류에 유리한' 표현으로 바뀐다.

즉 **사전 학습 표현이 목표 과제와 잘 맞지 않을수록 미세 조정의 이점이 커진다**.

## 연습 7-15

[도전 문제] [연습 문제 7-14]에서 데이터셋 구성을 바꿀 때 전이 학습용 훈련 데이터셋이 클래스별 100개씩 균일한 1,000개의 샘플로 구성되도록 데이터셋을 분리해 보자.

In [ ]:
# 클래스별 100개씩 균일하게 1,000개를 뽑는다.
train_set = mnist()
targets = train_set.targets
g = torch.Generator().manual_seed(SEED)
balanced = []
for c in range(10):
    idx = (targets == c).nonzero(as_tuple=True)[0]
    pick = idx[torch.randperm(len(idx), generator=g)[:100]]
    balanced.append(pick)
balanced = torch.cat(balanced)
print(f'선택한 샘플 {len(balanced)}개')
counts = torch.bincount(targets[balanced], minlength=10)
for c, n_c in enumerate(counts.tolist()):
    print(f'  클래스 {c}: {n_c}개')

클래스별로 인덱스를 모아 각각에서 100개씩 뽑으면 균일한 데이터셋이 된다. 무작위로 1,000개를 뽑으면 클래스별 개수가 80~120개로 흔들려, 적게 뽑힌 클래스의 정확도가 낮아지고 실험 비교가 어려워진다. 이런 분할을 **층화 추출**(stratified sampling)이라 한다.

## 연습 7-16

임베딩이나 오토인코더 등 데이터 속에 숨은 의미를 추출할 수 있는 모델만 전이 학습에 사용되지는 않는다. 일반적인 분류 모델이 학습한 저수준 특징 추출 능력을 활용하기 위한 전이 학습도 고려해 볼 만한 전략이다. 예를 들어 30x30 크기의 이미지에서 개와 고양이를 분류하도록 학습된 some_model이 있다고 하자. 이 모델을 재활용하면 새로운 작업, 즉 이미지 안에 동물이 있는지 없는지를 판단하는 모델을 만들 수 있다.

다음과 같은 구조로 정의되어 개와 고양이를 분류하도록 학습된 some_model을 재활용해, 이미지에 개 또는 고양이가 있는지 없는지를 판별하는 모델 클래스를 정의해 보자.

*코드 7-13 개와 고양이를 구분하기 위해 만든 다층 퍼셉트론 모델 some_model*

```python
import torch.nn as nn
some_model = nn.Sequential(
    nn.Linear(900, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)
```

### 풀이

30×30 이미지에서 개와 고양이를 분류하도록 학습된 `some_model`의 **앞쪽 합성곱 계층**은 경계·질감·색 대비 같은 **저수준 특징**을 추출한다. 이 특징은 개와 고양이에만 쓸모 있는 것이 아니라 대부분의 자연 이미지에 통한다.

따라서 다른 이미지 분류 과제에 전이할 때는 다음처럼 한다.

1. **앞쪽 계층은 그대로 가져와 고정**한다(저수준 특징 재사용).
2. **뒤쪽 계층과 분류기는 새 과제에 맞게 교체**한다. 출력 크기를 새 클래스 수로 바꾼다.
3. 데이터가 충분하다면 뒤쪽 합성곱 계층까지 **미세 조정**한다.

주의할 점은 **입력 형태와 전처리를 원래 모델과 똑같이 맞춰야** 한다는 것이다(30×30 크기, 같은 정규화). 8-4절의 ResNet-50 전이 학습이 바로 이 전략을 실제 모델에 적용한 예다.

## 연습 7-17

[도전 문제] 오토인코더를 재활용하는 전이 학습 예제를 소개했지만, 사실 임베딩이야말로 전이 학습의 단골손님이다. 자연어 텍스트를 학습한 임베딩은 같은 언어를 공유하는 문화권에서 만들어진 다른 자연어 텍스트에도 충분히 통한다.

7-2절의 예제를 통해 학습한 임베딩을 사용하는 전이 학습으로 소설 <이상한 나라의 앨리스>를 학습해서 소설을 쓰는 생성 모델에 도전해 보자. 소설 <오즈의 마법사>와 소설 <이상한 나라의 앨리스>의 텍스트를 합치는 방식으로 학습한 [연습 문제 7-8]과 또 다른 접근 방법이다.

그런데 오토인코더와 달리 임베딩의 가중치 파라미터를 재활용하는 전이 학습에는 두 텍스트에 포함된 토큰이 다를 수 있다는 문제가 숨어 있다. 이 경우 어휘 사전의 구성이 달라지고, 임베딩 행렬에서 각 행이 담당하는 토큰(단어)의 순서나 구성도 달라진다. 이 때문에 임베딩을 가지고 와서 사용하는 경우, 어휘 사전을 재구성하고 어휘 사전의 순서에 맞게 임베딩 행렬도 재구성해야 한다. 이 과정에 대한 설명은 7-4절의 깃허브 노트북 예제에 따로 정리해 두었다. 그리고 <오즈의 마법사>에는 없지만 <이상한 나라의 앨리스>에는 있는 단어도 처리해야 하므로 특징 추출 방법은 적절하지 않고 미세 조정 방식을 적용해야 한다.

이런 점들을 주의해서 사전 학습된 임베딩을 활용하는 전이 학습 모델을 학습해 보자. 이와 별도로 사전 학습 없이 <이상한 나라의 앨리스>만으로 처음부터 학습한 모델도 만든 후, 두 모델의 학습 과정 및 모델의 성능을 비교해 보자.

7장 학습 노트

In [ ]:
# 오즈의 마법사로 학습한 임베딩을 앨리스 텍스트 모델에 전이한다.
import re
def tokenize(text):
    text = text.lower().replace('\n', ' ')
    text = re.sub(r'([.,!?])', r' \1 ', text)
    return [t for t in text.split() if t]

def load_text(path):
    with open(path, encoding='utf-8-sig') as f:
        text = f.read()
    s, e = text.find('*** START'), text.find('*** END')
    if s != -1: text = text[text.find('\n', s) + 1:]
    if e != -1: text = text[:text.find('*** END')]
    return text

oz = tokenize(load_text('../../data/wonderful_wizard_of_oz.txt'))[:20000]
alice = tokenize(load_text('../../data/alice_in_wonderland.txt'))[:20000]
shared = sorted(set(oz) | set(alice))
vocab = {t: i for i, t in enumerate(shared)}
print(f'통합 어휘 {len(vocab):,}개 / 공통 단어 '
      f'{len(set(oz) & set(alice)):,}개')

class Writer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.fc(out[:, -1, :])

from torch.utils.data import TensorDataset
def make_loader(tokens, window=8):
    idx = torch.tensor([vocab[t] for t in tokens])
    xs = torch.stack([idx[i:i + window] for i in range(len(idx) - window)])
    return DataLoader(TensorDataset(xs, idx[window:]), batch_size=64, shuffle=True)

def train(model, loader, epochs=5):
    model = model.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=2e-3)
    for e in range(epochs):
        model.train(); t = n = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            t += loss.item() * len(y); n += len(y)
    return t / n

torch.manual_seed(SEED)
oz_model = Writer(len(vocab))
print(f'오즈 학습 손실 {train(oz_model, make_loader(oz)):.4f}')

import copy
for mode in ('전이 없음', '임베딩 전이(고정)', '임베딩 전이(미세 조정)'):
    torch.manual_seed(SEED)
    m2 = Writer(len(vocab))
    if '전이' in mode:
        m2.emb = copy.deepcopy(oz_model.emb)
        if '고정' in mode:
            m2.emb.weight.requires_grad = False
    print(f'{mode:22s} 앨리스 학습 손실 {train(m2, make_loader(alice)):.4f}')

두 소설이 같은 영어를 공유하므로 오즈에서 학습한 임베딩이 앨리스에도 통한다. 전이한 쪽이 같은 에포크에서 손실이 더 낮게 시작한다.

**고정**은 학습이 빠르지만 새 텍스트 고유의 단어를 반영하지 못하고, **미세 조정**은 느리지만 두 텍스트에 모두 맞는 표현을 만든다. 12장의 LLM 미세 조정도 규모만 다를 뿐 같은 원리다.